# Ollama + Cloudflare Tunnel: Serve Fine-Tuned Qwen Medical Validator from Colab

This notebook runs the fine-tuned **Qwen 2.5 3B** medical-validator model (GGUF Q4_K_M, ~1.8GB) on a Colab GPU via **Ollama**, then exposes it to your local dev environment through a **Cloudflare Tunnel**.

**How it works:**
1. Install Ollama on the Colab runtime
2. Load the GGUF model (from Google Drive or manual upload)
3. Create the Ollama model with the correct chat template
4. Expose the Ollama server via `cloudflared` tunnel
5. Set the printed URL as `QWEN_OLLAMA_URL` in your local `.env`

**Requirements:**
- Google Colab Pro (T4 or A100 GPU runtime)
- The `medical-validator-q4_k_m.gguf` file (~1.8GB) either on Google Drive or uploaded manually
- Runtime type set to **GPU** (Runtime > Change runtime type > T4 GPU)

In [ ]:
# ============================================================
# Configuration — edit these before running
# ============================================================

# Set True to mount Google Drive and copy the GGUF from there.
# Set False to upload the GGUF manually via the Colab file browser.
USE_GOOGLE_DRIVE = True

# Path to the GGUF on your Google Drive (only used if USE_GOOGLE_DRIVE=True)
GDRIVE_GGUF_PATH = "/content/drive/MyDrive/models/medical-validator-q4_k_m.gguf"

# Local working path — the GGUF will be copied/uploaded here
LOCAL_GGUF_PATH = "/content/medical-validator-q4_k_m.gguf"

# Ollama model name (must match what the worker expects)
MODEL_NAME = "medical-validator"

In [ ]:
## Cell 3: Install Ollama

!curl -fsSL https://ollama.com/install.sh | sh

# Verify installation
!ollama --version

In [ ]:
## Cell 4: Mount Google Drive / Load GGUF

import os
import shutil

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    if not os.path.exists(GDRIVE_GGUF_PATH):
        raise FileNotFoundError(
            f"GGUF not found at {GDRIVE_GGUF_PATH}\n"
            "Upload it to Google Drive first, or set USE_GOOGLE_DRIVE = False "
            "and upload manually via the Colab file browser."
        )

    print(f"Copying GGUF from Drive to {LOCAL_GGUF_PATH} ...")
    shutil.copy2(GDRIVE_GGUF_PATH, LOCAL_GGUF_PATH)
    size_mb = os.path.getsize(LOCAL_GGUF_PATH) / (1024 * 1024)
    print(f"Done. File size: {size_mb:.1f} MB")
else:
    print("Google Drive mount skipped.")
    print()
    print("Upload your GGUF manually:")
    print("  1. Click the folder icon in the left sidebar")
    print("  2. Click the upload button")
    print(f"  3. Upload 'medical-validator-q4_k_m.gguf' — it will land at {LOCAL_GGUF_PATH}")
    print()
    print("After uploading, run the next cell.")

# Final check
if os.path.exists(LOCAL_GGUF_PATH):
    size_mb = os.path.getsize(LOCAL_GGUF_PATH) / (1024 * 1024)
    print(f"\nGGUF ready at {LOCAL_GGUF_PATH} ({size_mb:.1f} MB)")
else:
    print(f"\nWaiting for GGUF at {LOCAL_GGUF_PATH} — upload it before continuing.")

In [ ]:
## Cell 5: Create Modelfile, start Ollama, and register the model

import subprocess, time, os

# Verify GGUF exists before proceeding
if not os.path.exists(LOCAL_GGUF_PATH):
    raise FileNotFoundError(
        f"GGUF not found at {LOCAL_GGUF_PATH}. "
        "Run the previous cell first (Drive mount or manual upload)."
    )

# Write the Modelfile — same template/params as training/ollama-model/Modelfile
modelfile_content = f"""FROM {LOCAL_GGUF_PATH}

PARAMETER temperature 0
PARAMETER num_ctx 2048
PARAMETER stop <|im_end|>

TEMPLATE \"\"\"{{{{ if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
<|im_start|>assistant
\"\"\"
"""

with open("/content/Modelfile", "w") as f:
    f.write(modelfile_content)

print("Modelfile written to /content/Modelfile")
print()

# Start ollama serve in background
print("Starting ollama serve ...")
proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama_serve.log", "w"),
    stderr=subprocess.STDOUT,
)

# Wait for server readiness (up to 30s)
import urllib.request, urllib.error
for i in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        print(f"Ollama server ready (took {i+1}s)")
        break
    except (urllib.error.URLError, ConnectionError):
        time.sleep(1)
else:
    raise RuntimeError("Ollama server failed to start within 30s. Check /content/ollama_serve.log")

print()

# Create the model
print(f"Creating model '{MODEL_NAME}' from Modelfile ...")
!ollama create {MODEL_NAME} -f /content/Modelfile

print()
print("Registered models:")
!ollama list

In [ ]:
## Cell 6: Test the model via OpenAI-compatible endpoint

import json, urllib.request

test_payload = json.dumps({
    "model": MODEL_NAME,
    "messages": [
        {
            "role": "system",
            "content": "You are a medical entity validator. Given company registration data, assess the risk level (LOW/MEDIUM/HIGH/UNKNOWN) and provide a brief summary."
        },
        {
            "role": "user",
            "content": json.dumps({
                "companyName": "Mayo Health System",
                "jurisdiction": "us_mn",
                "registrationNumber": "12345678",
                "incorporationDate": "1919-01-01",
                "legalStatus": "Active",
                "companyType": "Non-Profit Corporation"
            })
        }
    ],
    "temperature": 0
}).encode("utf-8")

req = urllib.request.Request(
    "http://localhost:11434/v1/chat/completions",
    data=test_payload,
    headers={"Content-Type": "application/json"},
)

try:
    with urllib.request.urlopen(req, timeout=60) as resp:
        result = json.loads(resp.read().decode())
        print("Status: OK")
        print()
        print(json.dumps(result, indent=2))
except Exception as e:
    print(f"Request failed: {e}")
    print("Check that ollama serve is running and the model was created successfully.")

In [ ]:
## Cell 7: Expose via Cloudflare Tunnel

import subprocess, time, re

# Download cloudflared
print("Downloading cloudflared ...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

print()
print("Starting Cloudflare tunnel to localhost:11434 ...")

# Start tunnel in background, capture output for URL extraction
tunnel_log = open("/content/cloudflared.log", "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434", "--no-autoupdate"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

# Wait for the tunnel URL to appear in logs (up to 30s)
tunnel_url = None
for i in range(30):
    time.sleep(1)
    try:
        with open("/content/cloudflared.log", "r") as f:
            log_content = f.read()
        match = re.search(r"(https://[a-zA-Z0-9\-]+\.trycloudflare\.com)", log_content)
        if match:
            tunnel_url = match.group(1)
            break
    except FileNotFoundError:
        pass

if tunnel_url:
    print()
    print("=" * 64)
    print(f"  Ollama is live! Set this in your .env:")
    print(f"  QWEN_OLLAMA_URL={tunnel_url}")
    print("=" * 64)
    print()
    print("Recommended env vars for the worker service:")
    print(f"  QWEN_OLLAMA_URL={tunnel_url}")
    print("  QWEN_CONCURRENCY=5")
    print("  QWEN_TIMEOUT_MS=15000")
    print()
    print(f"Test from your machine:")
    print(f"  curl {tunnel_url}/v1/models")
else:
    print("Tunnel URL not detected within 30s.")
    print("Check /content/cloudflared.log for errors:")
    !cat /content/cloudflared.log

## Keep Alive

Google Colab disconnects the runtime after ~30 minutes of idle time (or ~90 minutes for Pro users). This cell pings the Ollama server every 60 seconds to keep the connection active and prints a heartbeat so you can monitor uptime.

**Run this cell and leave the tab open.** Stop it manually (interrupt) when you are done.

In [ ]:
## Cell 8: Keep Alive — pings Ollama every 60s

import time, urllib.request, urllib.error
from datetime import datetime

print("Keepalive loop started. Interrupt this cell (stop button) to end.")
print()

ping_count = 0
while True:
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=5)
        status = "OK"
    except (urllib.error.URLError, ConnectionError, Exception) as e:
        status = f"FAIL ({e})"

    ping_count += 1
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] ping #{ping_count}: {status}")

    time.sleep(60)